## Raw data work

In [1]:
import gt_apps as my_apps

# 1. gtselect 설정
my_apps.filter['evclass'] = 1024       # GCE 분석을 위한 ULTRACLEANVETO 클래스
my_apps.filter['evtype'] = 3           # FRONT + BACK 데이터 모두 포함
my_apps.filter['ra'] = 266.4168        # 은하 중심 (Sgr A*) RA
my_apps.filter['dec'] = -29.0078       # 은하 중심 (Sgr A*) Dec
my_apps.filter['rad'] = 30             # 반경 30도
my_apps.filter['emin'] = 300           # 300 MeV (0.3 GeV) 이상
my_apps.filter['emax'] = 1000000       # 1 TeV 이하
my_apps.filter['zmax'] = 90            # 천정각 90도 컷 (대기 노이즈 차단)
my_apps.filter['tmin'] = 'INDEF'       # 다운로드 받은 시간 처음부터
my_apps.filter['tmax'] = 'INDEF'       # 끝까지
my_apps.filter['infile'] = 'events.txt'# 6개 파일이 묶인 리스트
my_apps.filter['outfile'] = 'GCE_17yr_filtered.fits'

print("Running gtselect... (데이터가 커서 몇 분 정도 소요될 수 있습니다)")
my_apps.filter.run()
print("gtselect 완료: GCE_17yr_filtered.fits 생성")

Running gtselect... (데이터가 커서 몇 분 정도 소요될 수 있습니다)
time -p gtselect infile=events.txt outfile=GCE_17yr_filtered.fits ra=266.4168 dec=-29.0078 rad=30.0 tmin="INDEF" tmax="INDEF" emin=300.0 emax=1000000.0 zmin=0.0 zmax=90.0 evclass=1024 evtype=3 convtype=-1 phasemin=0.0 phasemax=1.0 evtable="EVENTS" chatter=2 clobber=yes debug=no gui=no mode="ql"
Done.
real 174.34
user 161.23
sys 12.65
gtselect 완료: GCE_17yr_filtered.fits 생성


In [ ]:
# 2. gtmktime 설정
my_apps.maketime['scfile'] = 'L2602250837186609_SC00.fits'  # Spacecraft 파일
my_apps.maketime['filter'] = '(DATA_QUAL>0)&&(LAT_CONFIG==1)' # 공식 표준 품질 필터
my_apps.maketime['roicut'] = 'no'                             # zmax 컷은 gtselect에서 이미 정교하게 함
my_apps.maketime['evfile'] = 'GCE_17yr_filtered.fits'         # 이전 단계의 출력물
my_apps.maketime['outfile'] = 'GCE_17yr_gti.fits'

print("Running gtmktime... (GTI 계산 중)")
my_apps.maketime.run()
print("gtmktime 완료: GCE_17yr_gti.fits 생성")

Running gtmktime... (GTI 계산 중)
time -p gtmktime scfile=L2602250837186609_SC00.fits sctable="SC_DATA" filter="(DATA_QUAL>0)&&(LAT_CONFIG==1)" roicut=no evfile=GCE_17yr_filtered.fits evtable="EVENTS" outfile="GCE_17yr_gti.fits" apply_filter=yes overwrite=no header_obstimes=yes tstart=0.0 tstop=0.0 gtifile="default" chatter=2 clobber=yes debug=no gui=no mode="ql"
real 683.20
user 674.24
sys 7.52
gtmktime 완료: GCE_17yr_gti.fits 생성


: 

In [ ]:
import gt_apps as my_apps

# =====================================================================
# 1. 3D Counts Map (CCUBE) 생성
# =====================================================================
my_apps.evtbin['algorithm'] = 'CCUBE'
my_apps.evtbin['evfile'] = 'GCE_17yr_gti.fits'              # 이전 단계에서 만든 깨끗한 데이터
my_apps.evtbin['outfile'] = 'GCE_17yr_ccube.fits'
my_apps.evtbin['scfile'] = 'L2602250837186609_SC00.fits'    # 우주선(Spacecraft) 파일

# 공간 Binning 설정 (40x40도 ROI 생성)
my_apps.evtbin['nxpix'] = 400       # 가로 픽셀 수 (400 * 0.1도 = 40도)
my_apps.evtbin['nypix'] = 400       # 세로 픽셀 수 (400 * 0.1도 = 40도)
my_apps.evtbin['binsz'] = 0.1       # 픽셀당 0.1도 (GCE 분석의 표준 해상도)
my_apps.evtbin['coordsys'] = 'GAL'  # 은하 좌표계
my_apps.evtbin['xref'] = 0.0        # 은하 중심 (l=0)
my_apps.evtbin['yref'] = 0.0        # 은하 중심 (b=0)
my_apps.evtbin['axisrot'] = 0.0
my_apps.evtbin['proj'] = 'CAR'      # 평면 투영 (Cartesian)

# 에너지 Binning 설정
my_apps.evtbin['ebinalg'] = 'LOG'
my_apps.evtbin['emin'] = 300        # 300 MeV
my_apps.evtbin['emax'] = 1000000    # 1 TeV
my_apps.evtbin['enumbins'] = 17     # 이전 실습과 동일하게 14개 또는 17개로 설정

print("1. Running gtbin (CCUBE 생성 중...)")
my_apps.evtbin.run()
print("✅ GCE_17yr_ccube.fits 생성 완료!")

# =====================================================================
# 2. Livetime Cube 생성 (가장 오래 걸리는 작업)
# =====================================================================
my_apps.expCube['evfile'] = 'GCE_17yr_gti.fits'
my_apps.expCube['scfile'] = 'L2602250837186609_SC00.fits'
my_apps.expCube['outfile'] = 'GCE_17yr_ltcube.fits'
my_apps.expCube['dcostheta'] = 0.025
my_apps.expCube['binsz'] = 1        # 기본값 1도 (라이브타임 큐브는 1도로 충분합니다)
my_apps.expCube['zmax'] = 90        # gtselect에서 설정한 zenith cut과 반드시 동일해야 함

print("\n2. Running gtltcube (Livetime 계산 중...)")
print("⚠️ 주의: 17년 치 데이터의 궤도를 적분하는 작업이므로 CPU 성능에 따라 수 시간~하루 이상 걸릴 수 있습니다.")
my_apps.expCube.run()
print("✅ GCE_17yr_ltcube.fits 생성 완료!")

1. Running gtbin (CCUBE 생성 중...)
time -p gtbin evfile=GCE_17yr_gti.fits scfile=L2602250837186609_SC00.fits outfile=GCE_17yr_ccube.fits algorithm="CCUBE" ebinalg="LOG" emin=300.0 emax=1000000.0 enumbins=17 ebinfile=NONE tbinalg="LIN" tbinfile=NONE nxpix=400 nypix=400 binsz=0.1 coordsys="GAL" xref=0.0 yref=0.0 axisrot=0.0 rafield="RA" decfield="DEC" proj="CAR" hpx_ordering_scheme="RING" hpx_order=3 hpx_ebin=yes hpx_region="" evtable="EVENTS" sctable="SC_DATA" efield="ENERGY" tfield="TIME" chatter=2 clobber=yes debug=no gui=no mode="ql"
This is gtbin version HEAD
real 16.33
user 15.11
sys 1.18
✅ GCE_17yr_ccube.fits 생성 완료!

2. Running gtltcube (Livetime 계산 중...)
⚠️ 주의: 17년 치 데이터의 궤도를 적분하는 작업이므로 CPU 성능에 따라 수 시간~하루 이상 걸릴 수 있습니다.
time -p gtltcube evfile="GCE_17yr_gti.fits" evtable="EVENTS" scfile=L2602250837186609_SC00.fits sctable="SC_DATA" outfile=GCE_17yr_ltcube.fits dcostheta=0.025 binsz=1.0 phibins=0 tmin=0.0 tmax=0.0 file_version="1" zmin=0.0 zmax=90.0 chatter=2 clobber=yes debug=no gui

: 

In [1]:
%%bash
echo "Running gtexpcube2 (노출 지도 계산 중...)"

# Binned Exposure Cube 생성
gtexpcube2 \
    infile=GCE_17yr_ltcube.fits \
    cmap=GCE_17yr_ccube.fits \
    outfile=GCE_17yr_expcube.fits \
    irfs=P8R3_ULTRACLEANVETO_V3 \
    evtype=3 \
    chatter=2

echo "✅ GCE_17yr_expcube.fits 생성 완료!"

Running gtexpcube2 (노출 지도 계산 중...)


Computing binned exposure map....................!


Using evtype=3 (i.e., FRONT/BACK irfs)
✅ GCE_17yr_expcube.fits 생성 완료!


## Generate XML Model file

In [ ]:
# download the FL16Y (from https://fermi.gsfc.nasa.gov/ssc/data/access/lat/fl16y/)

In [10]:
import os
import xml.etree.ElementTree as ET

# SourceList.py 파일 안의 SourceList 클래스를 직접 불러옵니다
from SourceList import SourceList 

# =====================================================================
# 1. 파일명 설정
# =====================================================================
catalog_file = 'gll_psc_v40.fit'                     

# [⭐핵심 수정⭐] ccube가 아닌 gti.fits(이벤트 리스트)를 넣어야 헤더를 정상적으로 읽습니다!
roi_file = 'GCE_17yr_gti.fits'                     

iso_file = 'iso_P8R3_ULTRACLEANVETO_V3_v1.txt'       
dummy_gal_file = 'gll_iem_v07.fits'                  
extended_dir = 'LAT_extended_sources_16years'

base_xml = 'GCE_17yr_FL16Y_base.xml'

# NAMING CONVENTION 파일 및 GDE 맵 폴더 경로
naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
gde_dir = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/'

# =====================================================================
# 2. 기본 XML 뼈대 생성
# =====================================================================
if not os.path.exists(base_xml):
    print("FL16Y 카탈로그를 로드하여 기본 XML 뼈대를 생성합니다 (최초 1회)...")
    
    # SourceList 초기화 (FL16Y 카탈로그 양식에 맞게 DR=4 로 지정)
    mymodel = SourceList(catalog_file, roi_file, base_xml, DR=4)
    
    # make_model 함수 호출 시 extended_directory 매개변수로 확장 선원 경로 전달
    mymodel.make_model(galactic_file=dummy_gal_file, 
                       galactic_name='gll_iem', 
                       isotropic_file=iso_file, 
                       isotropic_name='iso_p8v3',
                       extended_directory=extended_dir)
else:
    print(f"✅ 기존에 생성된 기본 뼈대({base_xml})를 재사용합니다.")

# =====================================================================
# 3. 모델 이름 규칙 파일 파싱 (80개 모델 목록 추출)
# =====================================================================
models_info = []
with open(naming_conv_file, 'r') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.split()
        if len(parts) >= 2:
            roman_name = parts[0]   
            folder_code = parts[1]  
            models_info.append((roman_name, folder_code))

print(f"총 {len(models_info)}개의 GDE 모델 XML을 일괄 생성합니다...\n")

# =====================================================================
# 4. 80개 모델에 대한 XML 자동 생성 루프
# =====================================================================
for roman_name, folder_code in models_info:
    final_xml = f'GCE_17yr_FL16Y_Model_{roman_name}.xml'
    
    # 뼈대 XML 파싱
    tree = ET.parse(base_xml)
    root = tree.getroot()

    # 더미 은하 배경(gll_iem) 블록 찾아서 삭제
    for source in root.findall('source'):
        if source.get('name') == 'gll_iem':
            root.remove(source)

    # 현재 루프의 모델(folder_code)에 맞는 GDE 맵 3종 세팅
    gde_components = [
        {'name': f'Pi0_Model{roman_name}',    'file': os.path.join(gde_dir, f'pi0_{folder_code}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')},
        {'name': f'Bremss_Model{roman_name}', 'file': os.path.join(gde_dir, f'bremss_{folder_code}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')},
        {'name': f'ICS_Model{roman_name}',    'file': os.path.join(gde_dir, f'ICS_{folder_code}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')}
    ]

    # XML에 GDE 성분 주입
    for comp in gde_components:
        source = ET.SubElement(root, 'source', {'name': comp['name'], 'type': 'DiffuseSource'})
        
        spectrum = ET.SubElement(source, 'spectrum', {'type': 'ConstantValue'})
        ET.SubElement(spectrum, 'parameter', {'name': 'Value', 'value': '1.0', 'free': '1', 'max': '10.0', 'min': '0.1', 'scale': '1.0'})
        
        spatial = ET.SubElement(source, 'spatialModel', {'type': 'SpatialMap', 'file': comp['file']})
        ET.SubElement(spatial, 'parameter', {'name': 'Prefactor', 'value': '1.0', 'free': '0', 'max': '1000.0', 'min': '0.001', 'scale': '1.0'})

    # 최종 XML 저장
    tree.write(final_xml, encoding='utf-8', xml_declaration=True)
    
print("🎉 모든 GDE 모델(80개)에 대한 맞춤형 XML 파일 생성이 완료되었습니다!")

✅ 기존에 생성된 기본 뼈대(GCE_17yr_FL16Y_base.xml)를 재사용합니다.
총 80개의 GDE 모델 XML을 일괄 생성합니다...

🎉 모든 GDE 모델(80개)에 대한 맞춤형 XML 파일 생성이 완료되었습니다!


In [1]:
import os
import glob
import xml.etree.ElementTree as ET
from astropy.io import fits

# =====================================================================
# 1. XML 일괄 패치 (SpatialMap -> MapCubeFunction)
# =====================================================================
print("1. XML 모델들의 태그를 3D 큐브용(MapCubeFunction)으로 수정합니다...")
xml_files = glob.glob('XML_models/*.xml')
count = 0

for xf in xml_files:
    tree = ET.parse(xf)
    root = tree.getroot()
    changed = False
    
    for source in root.findall('source'):
        spatial = source.find('spatialModel')
        # 기존에 SpatialMap으로 잘못 들어간 태그를 MapCubeFunction으로 변경
        if spatial is not None and spatial.get('type') == 'SpatialMap':
            if any(x in source.get('name') for x in ['Pi0', 'Bremss', 'ICS']):
                spatial.set('type', 'MapCubeFunction')
                changed = True
                
    if changed:
        tree.write(xf, encoding='utf-8', xml_declaration=True)
        count += 1

print(f"✅ 총 {count}개의 XML 파일 수정 완료!\n")

# =====================================================================
# 2. 에러가 발생한 FITS 파일 헤더(WCS) 진단
# =====================================================================
print("2. 문제가 된 FITS 파일의 내부 헤더를 점검합니다...")
test_file = "/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/bremss_5q_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits"

try:
    with fits.open(test_file) as hdul:
        print(f"👉 이 파일은 총 {len(hdul)}개의 HDU(데이터 블록)를 가지고 있습니다.")
        for i in range(len(hdul)):
            if 'CTYPE1' in hdul[i].header:
                print(f"  - HDU {i}: CTYPE1 = '{hdul[i].header['CTYPE1']}' (정상 좌표계 존재 🟢)")
            else:
                print(f"  - HDU {i}: CTYPE1 없음 (좌표계 누락 ❌)")
except Exception as e:
    print(f"파일을 읽는 중 에러 발생: {e}")

1. XML 모델들의 태그를 3D 큐브용(MapCubeFunction)으로 수정합니다...
✅ 총 80개의 XML 파일 수정 완료!

2. 문제가 된 FITS 파일의 내부 헤더를 점검합니다...
👉 이 파일은 총 1개의 HDU(데이터 블록)를 가지고 있습니다.
  - HDU 0: CTYPE1 없음 (좌표계 누락 ❌)


In [2]:
import os
import glob
from astropy.io import fits

# 1. Zenodo GDE 맵들이 저장된 폴더 경로
gde_dir = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/'

# 폴더 내의 모든 fits 파일 검색
fits_files = glob.glob(os.path.join(gde_dir, '*.fits'))

print(f"총 {len(fits_files)}개의 GDE 맵 파일에 WCS 좌표계 헤더 주입을 시작합니다...\n")

success_count = 0
naxis_value = None

for f in fits_files:
    try:
        # 'update' 모드로 열어서 수정 후 바로 저장되게 함
        with fits.open(f, mode='update') as hdul:
            header = hdul[0].header
            
            # 차원 수(2D인지 3D인지) 확인용
            if naxis_value is None:
                naxis_value = header.get('NAXIS', 2)
            
            # 이미 CTYPE1이 'GLON-CAR'로 정상 주입되어 있다면 건너뜀
            if 'CTYPE1' in header and header['CTYPE1'] == 'GLON-CAR':
                continue
            
            # 데이터의 실제 픽셀 개수 가져오기 (보통 240x240)
            naxis1 = header.get('NAXIS1', 240) # 가로 픽셀 수
            naxis2 = header.get('NAXIS2', 240) # 세로 픽셀 수
            
            # =======================================================
            # [핵심] Fermi-LAT 표준 은하 좌표계(Cartesian) 강제 주입
            # =======================================================
            header['CTYPE1'] = 'GLON-CAR'                # X축: 은경 (Galactic Longitude)
            header['CUNIT1'] = 'deg'                     # 단위: 각도(degree)
            header['CRVAL1'] = 0.0                       # 중심점 좌표: 0도
            header['CRPIX1'] = naxis1 / 2.0 + 0.5        # 중심점 픽셀 위치 (120.5)
            header['CDELT1'] = -0.25                     # 픽셀 크기 (-0.25도, 은경은 오른쪽에서 왼쪽으로 감소)
            
            header['CTYPE2'] = 'GLAT-CAR'                # Y축: 은위 (Galactic Latitude)
            header['CUNIT2'] = 'deg'                     # 단위: 각도(degree)
            header['CRVAL2'] = 0.0                       # 중심점 좌표: 0도
            header['CRPIX2'] = naxis2 / 2.0 + 0.5        # 중심점 픽셀 위치 (120.5)
            header['CDELT2'] = 0.25                      # 픽셀 크기 (+0.25도)
            
            # 만약 3차원(에너지 축 포함) 데이터라면 에너지 헤더도 기본값으로 추가
            if header.get('NAXIS', 2) >= 3:
                header['CTYPE3'] = 'Energy'
                header['CUNIT3'] = 'MeV'
            
            hdul.flush() # 변경사항 디스크에 즉시 쓰기
            success_count += 1
            
    except Exception as e:
        print(f"❌ 에러 발생 ({os.path.basename(f)}): {e}")

print(f"✅ {success_count}개의 파일에 완벽하게 좌표계 이식을 완료했습니다!")
print(f"🧐 (참고: 이 데이터는 {naxis_value}D 데이터 배열입니다.)")

총 241개의 GDE 맵 파일에 WCS 좌표계 헤더 주입을 시작합니다...

✅ 241개의 파일에 완벽하게 좌표계 이식을 완료했습니다!
🧐 (참고: 이 데이터는 3D 데이터 배열입니다.)


In [1]:
import glob
import xml.etree.ElementTree as ET

print("XML 파일들의 3D 큐브 파라미터 이름(Prefactor -> Normalization)을 수정합니다...")

xml_files = glob.glob('XML_models/*.xml')
count = 0

for xf in xml_files:
    tree = ET.parse(xf)
    root = tree.getroot()
    changed = False
    
    for source in root.findall('source'):
        spatial = source.find('spatialModel')
        # MapCubeFunction인 경우 내부 파라미터 확인
        if spatial is not None and spatial.get('type') == 'MapCubeFunction':
            for param in spatial.findall('parameter'):
                # Prefactor로 잘못 적혀있다면 Normalization으로 강제 변경
                if param.get('name') == 'Prefactor':
                    param.set('name', 'Normalization')
                    changed = True
                    
    if changed:
        tree.write(xf, encoding='utf-8', xml_declaration=True)
        count += 1

print(f"✅ 총 {count}개의 XML 파일 수정 완료! 이제 완벽합니다.")

XML 파일들의 3D 큐브 파라미터 이름(Prefactor -> Normalization)을 수정합니다...
✅ 총 80개의 XML 파일 수정 완료! 이제 완벽합니다.


In [1]:
import os
import glob
import numpy as np
from astropy.io import fits

# 1. Zenodo GDE 맵들이 저장된 폴더 경로
gde_dir = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/'
fits_files = glob.glob(os.path.join(gde_dir, '*.fits'))

print(f"총 {len(fits_files)}개의 GDE 맵 파일에 [ENERGIES] 확장 테이블 주입을 시작합니다...\n")

success_count = 0
skip_count = 0

for f in fits_files:
    try:
        # update 모드로 열기
        with fits.open(f, mode='update') as hdul:
            # 이미 ENERGIES 테이블이 존재하면 건너뜀
            if len(hdul) > 1 and any(hdu.name == 'ENERGIES' for hdu in hdul):
                skip_count += 1
                continue
            
            header = hdul[0].header
            
            # Z축(에너지 축)의 층수(빈 개수) 가져오기
            n_bins = header.get('NAXIS3', None)
            if n_bins is None:
                print(f"⚠️ NAXIS3(에너지 축) 정보가 없습니다: {os.path.basename(f)}")
                continue
            
            # 파일명에서 힌트를 얻은 에너지 대역 (50 MeV ~ 814008 MeV)
            # Fermi-LAT은 보통 에너지를 로그 스케일(Log-space)로 나눕니다.
            energies = np.logspace(np.log10(50.0), np.log10(814008.0), n_bins)
            
            # ENERGIES 테이블 형식 생성 (컬럼명: Energy, 단위: MeV, 포맷: D(Double))
            col = fits.Column(name='Energy', format='D', unit='MeV', array=energies)
            cols = fits.ColDefs([col])
            
            # BinTableHDU 객체 생성
            ext_hdu = fits.BinTableHDU.from_columns(cols, name='ENERGIES')
            
            # 기존 FITS 파일에 테이블 블록 덧붙이기(Append)
            hdul.append(ext_hdu)
            hdul.flush() # 디스크에 저장
            
            success_count += 1
            
    except Exception as e:
        print(f"❌ 에러 발생 ({os.path.basename(f)}): {e}")

print(f"✅ {success_count}개의 파일에 [ENERGIES] 테이블 주입 완료! (건너뜀: {skip_count}개)")

총 241개의 GDE 맵 파일에 [ENERGIES] 확장 테이블 주입을 시작합니다...

✅ 241개의 파일에 [ENERGIES] 테이블 주입 완료! (건너뜀: 0개)


In [1]:
%%bash
echo "넓은 경계를 가진 100x100도 노출 지도를 새로 계산합니다..."

gtexpcube2 \
    infile=GCE_17yr_ltcube.fits \
    cmap=none \
    outfile=GCE_17yr_expcube_large.fits \
    irfs=P8R3_ULTRACLEANVETO_V3 \
    evtype=3 \
    nxpix=200 \
    nypix=200 \
    binsz=0.5 \
    coordsys=GAL \
    xref=0 yref=0 proj=CAR \
    emin=300 \
    emax=1000000 \
    enumbins=17 \
    chatter=2

echo "✅ GCE_17yr_expcube_large.fits 생성 완료!"

넓은 경계를 가진 100x100도 노출 지도를 새로 계산합니다...
Rotation angle of image axis, in degrees[0] 

Computing binned exposure map....................!


Using evtype=3 (i.e., FRONT/BACK irfs)


In [1]:
import glob
import xml.etree.ElementTree as ET

print("XML 파일 내의 확장 선원(Extended Sources) 템플릿 경로를 수정합니다...")

xml_files = glob.glob('XML_models/*.xml')
count = 0

for xf in xml_files:
    tree = ET.parse(xf)
    root = tree.getroot()
    changed = False
    
    for source in root.findall('source'):
        spatial = source.find('spatialModel')
        if spatial is not None:
            file_path = spatial.get('file')
            # 기존에 'LAT_extended_sources_16years/파일명.fits' 로 잘못 적힌 경로 찾기
            if file_path and 'LAT_extended_sources_16years' in file_path and 'Templates' not in file_path:
                
                # 파일 이름만 추출 (예: HESSJ1841-055.fits)
                file_name = file_path.split('/')[-1]
                
                # 올바른 경로(Templates 폴더 포함)로 수정
                new_path = f"LAT_extended_sources_16years/Templates/{file_name}"
                spatial.set('file', new_path)
                changed = True
                
    if changed:
        tree.write(xf, encoding='utf-8', xml_declaration=True)
        count += 1

print(f"✅ 총 {count}개의 XML 파일 경로 수정 완료! 이제 진짜 완벽합니다.")

XML 파일 내의 확장 선원(Extended Sources) 템플릿 경로를 수정합니다...
✅ 총 80개의 XML 파일 경로 수정 완료! 이제 진짜 완벽합니다.


In [ ]:
import os
import gt_apps as my_apps

# =====================================================================
# 1. 파일 및 폴더 설정
# =====================================================================
xml_dir = 'XML_models'                 
srcmap_dir = 'Source_Maps'             

if not os.path.exists(srcmap_dir):
    os.makedirs(srcmap_dir)

naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
models_info = []
with open(naming_conv_file, 'r') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.split()
        if len(parts) >= 2:
            models_info.append(parts[0])

print("▶️ 확장된 노출 지도를 사용하여 Source Maps 자동 생성을 재시작합니다...\n")

# =====================================================================
# 2. Source Maps 자동 생성 루프
# =====================================================================
for roman_name in models_info:
    xml_file = os.path.join(xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}.xml')
    outfile = os.path.join(srcmap_dir, f'GCE_17yr_srcmap_Model_{roman_name}.fits')
    
    # 이미 1MB 이상으로 잘 구워진 파일은 건너뜁니다!
    if os.path.exists(outfile) and os.path.getsize(outfile) > 1000000:
        print(f"⏩ 이미 정상 생성됨: {outfile} (건너뜀)")
        continue
        
    print(f"▶️ 생성 중: {outfile} ...")
    
    my_apps.srcMaps['expcube'] = 'GCE_17yr_ltcube.fits'          
    my_apps.srcMaps['cmap']    = 'GCE_17yr_ccube.fits'              
    my_apps.srcMaps['srcmdl']  = xml_file           
    
    # [수정된 부분] 방금 새로 만든 넉넉한 노출 지도를 사용합니다!
    my_apps.srcMaps['bexpmap'] = 'GCE_17yr_expcube_large.fits'          
    my_apps.srcMaps['outfile'] = outfile  
    my_apps.srcMaps['irfs']    = 'P8R3_ULTRACLEANVETO_V3'           
    my_apps.srcMaps['evtype']  = 3                                
    my_apps.srcMaps['chatter'] = 0

    try:
        my_apps.srcMaps.run()
        print(f"  ✅ 완료: Model {roman_name}")
    except Exception as e:
        print(f"  ❌ 에러 발생 (Model {roman_name}): {e}")

print("\n🎉 모든 소스 맵(80개) 생성이 완료되었습니다!")

▶️ 확장된 노출 지도를 사용하여 Source Maps 자동 생성을 재시작합니다...

▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_I.fits ...
  ✅ 완료: Model I
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_II.fits ...
  ✅ 완료: Model II
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_III.fits ...
  ✅ 완료: Model III
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_IV.fits ...
  ✅ 완료: Model IV
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_V.fits ...
  ✅ 완료: Model V
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_VI.fits ...
  ✅ 완료: Model VI
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_VII.fits ...
  ✅ 완료: Model VII
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_VIII.fits ...
  ✅ 완료: Model VIII
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_IX.fits ...
  ✅ 완료: Model IX
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_X.fits ...
  ✅ 완료: Model X
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_XI.fits ...
  ✅ 완료: Model XI
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_XII.fits ...
  ✅ 완료: Model XII
▶️ 생성 중: Source_Maps/GCE_17yr_srcmap_Model_XIII.fits ...
  ✅ 완료: Model XIII
▶️ 생성 중

KeyboardInterrupt: 

: 

In [ ]:
# do not run in jupyter

import os
import multiprocessing as mp
import time

def run_perfect_srcmaps(roman_name):
    import gt_apps as my_apps
    
    xml_dir = 'XML_models'
    srcmap_dir = 'Source_Maps_Perfect' # 흠집 없는 완벽한 맵이 저장될 새 폴더
    
    if not os.path.exists(srcmap_dir):
        os.makedirs(srcmap_dir, exist_ok=True)
        
    xml_file = os.path.join(xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}.xml')
    outfile = os.path.join(srcmap_dir, f'GCE_17yr_srcmap_Model_{roman_name}.fits')
    
    # 완벽한 맵은 점광원과 배경이 모두 포함되어 용량이 큽니다 (약 30MB 이상)
    if os.path.exists(outfile) and os.path.getsize(outfile) > 30000000:
        return (roman_name, "이미 존재함 (건너뜀)")
        
    try:
        my_apps.srcMaps['expcube'] = 'GCE_17yr_ltcube.fits'          
        my_apps.srcMaps['cmap']    = 'GCE_17yr_ccube.fits'              
        my_apps.srcMaps['srcmdl']  = xml_file           
        my_apps.srcMaps['bexpmap'] = 'GCE_17yr_expcube_large.fits'         
        my_apps.srcMaps['outfile'] = outfile  
        my_apps.srcMaps['irfs']    = 'P8R3_ULTRACLEANVETO_V3'           
        my_apps.srcMaps['evtype']  = 3                                
        
        # ⭐ [가장 중요한 핵심] 점광원까지 모조리 FITS 파일 안에 구워 넣습니다!
        my_apps.srcMaps['ptsrc']   = 'yes'  
        my_apps.srcMaps['chatter'] = 0

        my_apps.srcMaps.run()
        return (roman_name, "✅ 생성 성공")
    except Exception as e:
        return (roman_name, f"❌ 에러 발생: {e}")

if __name__ == '__main__':
    start_time = time.time()
    
    naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
    models_info = []
    with open(naming_conv_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 2:
                models_info.append(parts[0])

    # 64코어 서버의 위력을 발휘합니다. (소스맵 생성은 메모리를 덜 먹으므로 32코어 동시 실행 가능)
    NUM_CORES = 32
    print(f"▶️ 80개 전체 모델의 '완벽한 Source Maps' 병렬 생성을 시작합니다. (코어: {NUM_CORES})")
    print("--------------------------------------------------")

    with mp.Pool(processes=NUM_CORES) as pool:
        for result in pool.imap_unordered(run_perfect_srcmaps, models_info):
            print(f"Model {result[0]} : {result[1]}")

    end_time = time.time()
    print(f"\n🎉 완벽한 소스 맵 병렬 생성 완료! (소요 시간: {(end_time - start_time)/3600:.2f} 시간)")

In [ ]:
import os
from BinnedAnalysis import *

# =====================================================================
# 1. 폴더 및 파일 설정
# =====================================================================
xml_dir = 'XML_models'
srcmap_dir = 'Source_Maps'
fit_xml_dir = 'Fitted_XML_models' # 피팅이 완료된 파라미터가 저장될 새 폴더

if not os.path.exists(fit_xml_dir):
    os.makedirs(fit_xml_dir)

naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
models_info = []
with open(naming_conv_file, 'r') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.split()
        if len(parts) >= 2:
            models_info.append(parts[0])

# 최종 결과(우도 값)를 저장할 CSV 파일
results_file = 'Likelihood_Results_17yr.csv'
with open(results_file, 'w') as f:
    f.write("Model,LogLikelihood\n")

print("▶️ 80개 모델에 대한 Binned Likelihood Fitting을 시작합니다...\n")

# =====================================================================
# 2. 80개 모델 자동 피팅 루프
# =====================================================================
for roman_name in models_info:
    xml_file = os.path.join(xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}.xml')
    srcmap_file = os.path.join(srcmap_dir, f'GCE_17yr_srcmap_Model_{roman_name}.fits')
    out_xml_file = os.path.join(fit_xml_dir, f'GCE_17yr_fitted_Model_{roman_name}.xml')

    if not os.path.exists(srcmap_file):
        print(f"⚠️ {srcmap_file} 이(가) 없어 건너뜁니다.")
        continue

    print(f"========================================")
    print(f"🚀 피팅 시작: Model {roman_name}")
    print(f"========================================")

    try:
        # 1. BinnedObs 객체 로드 (데이터, 소스맵, 노출지도 묶기)
        obs = BinnedObs(srcMaps=srcmap_file,
                        expCube='GCE_17yr_ltcube.fits',
                        binnedExpMap='GCE_17yr_expcube_large.fits',
                        irfs='P8R3_ULTRACLEANVETO_V3')

        # 2. Likelihood 최적화 객체 생성 (NewMinuit 알고리즘 사용)
        like = BinnedAnalysis(obs, srcModel=xml_file, optimizer='NewMinuit')

        # 3. 우도 피팅 수행 (서버 성능에 따라 모델당 십여 분~수 시간 소요)
        likeObj = like.fit(covar=True, tol=1e-2, verbosity=0)

        # 4. 피팅된 최종 Log-Likelihood 값 추출
        logL = like.logLike.value()
        print(f"  ✅ 피팅 성공! -log(L) = {logL}")

        # 5. 최적화된 변수들이 기록된 새 XML 파일 저장
        like.logLike.writeXml(out_xml_file)

        # 6. CSV 파일에 결과 기록 (중간에 끊겨도 데이터가 남도록 바로바로 저장)
        with open(results_file, 'a') as f:
            f.write(f"{roman_name},{logL}\n")

    except Exception as e:
        print(f"  ❌ 에러 발생 (Model {roman_name}): {e}")

print("\n🎉 17년 치 데이터의 모든 우도 피팅이 완료되었습니다! CSV 결과를 확인하세요.")

In [ ]:
# do not run in jupyter

import os
import multiprocessing as mp
import time

# =====================================================================
# 1. 단일 모델 피팅을 수행하는 작업자(Worker) 함수
# =====================================================================
def run_fitting_for_model(roman_name):
    """
    각 코어가 독립적으로 실행할 함수입니다.
    SWIG/C++ 기반의 Fermi 도구들이 멀티프로세싱과 충돌하지 않도록 
    import BinnedAnalysis를 함수 안에서 독립적으로 수행합니다.
    """
    import BinnedAnalysis as ba
    
    xml_dir = 'XML_models'
    srcmap_dir = 'Source_Maps'
    fit_xml_dir = 'Fitted_XML_models'
    
    xml_file = os.path.join(xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}.xml')
    srcmap_file = os.path.join(srcmap_dir, f'GCE_17yr_srcmap_Model_{roman_name}.fits')
    out_xml_file = os.path.join(fit_xml_dir, f'GCE_17yr_fitted_Model_{roman_name}.xml')
    
    if not os.path.exists(srcmap_file):
        return (roman_name, None, "Source map 파일 누락")
        
    try:
        # 1. 관측 데이터 로드
        obs = ba.BinnedObs(srcMaps=srcmap_file,
                           expCube='GCE_17yr_ltcube.fits',
                           binnedExpMap='GCE_17yr_expcube_large.fits',
                           irfs='P8R3_ULTRACLEANVETO_V3')
                           
        # 2. 피팅 객체 생성
        like = ba.BinnedAnalysis(obs, srcModel=xml_file, optimizer='NewMinuit')
        
        # 3. 우도 피팅 수행 (verbosity=0으로 터미널 로그 최소화)
        likeObj = like.fit(covar=True, tol=1e-2, verbosity=0)
        
        # 4. 피팅된 파라미터 저장 및 로그우도 반환
        logL = like.logLike.value()
        like.logLike.writeXml(out_xml_file)
        
        return (roman_name, logL, "Success")
        
    except Exception as e:
        return (roman_name, None, f"Error: {e}")

# =====================================================================
# 2. 메인 실행 블록 (멀티프로세싱 코어 할당 및 관리)
# =====================================================================
if __name__ == '__main__':
    start_time = time.time()
    
    xml_dir = 'XML_models'
    srcmap_dir = 'Source_Maps'
    fit_xml_dir = 'Fitted_XML_models'

    if not os.path.exists(fit_xml_dir):
        os.makedirs(fit_xml_dir)

    # 80개 모델 목록 불러오기
    naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
    models_info = []
    with open(naming_conv_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 2:
                models_info.append(parts[0])

    # RAM(128GB)을 고려하여 동시 실행 코어 수를 32개로 안전하게 제한
    NUM_CORES = 32  
    print(f"▶️ 총 {len(models_info)}개의 모델에 대해 {NUM_CORES}개의 코어를 이용한 병렬 피팅을 시작합니다...")

    results_file = 'Likelihood_Results_17yr.csv'
    with open(results_file, 'w') as f:
        f.write("Model,LogLikelihood,Status\n")

    # multiprocessing Pool을 이용한 병렬 처리 시작
    with mp.Pool(processes=NUM_CORES) as pool:
        # imap_unordered를 쓰면 먼저 끝난 작업부터 즉각적으로 반환받습니다.
        for result in pool.imap_unordered(run_fitting_for_model, models_info):
            roman_name, logL, status = result
            
            # 하나 완료될 때마다 터미널에 출력 및 CSV에 즉시 저장
            if status == "Success":
                print(f" ✅ [완료] Model {roman_name} | -log(L) = {logL:.2f}")
            else:
                print(f" ❌ [실패] Model {roman_name} | 이유: {status}")
                
            with open(results_file, 'a') as f:
                f.write(f"{roman_name},{logL},{status}\n")

    end_time = time.time()
    print(f"\n🎉 병렬 피팅 대장정이 모두 완료되었습니다!")
    print(f"⏱️ 총 소요 시간: {(end_time - start_time)/3600:.2f} 시간")

In [ ]:
# do not run in jupyter
# multi-core likelihood code merge-fixed version

import os
import multiprocessing as mp
import time

# =====================================================================
# 1. 단일 모델 피팅을 수행하는 작업자(Worker) 함수
# =====================================================================
def run_fitting_for_model(roman_name):
    """
    고정된 배경 소스를 병합(Merge)하여 메모리 사용량을 획기적으로 줄이고
    피팅 속도를 비약적으로 상승시키는 최적화된 워커 함수입니다.
    """
    import BinnedAnalysis as ba
    import xml.etree.ElementTree as ET
    from astropy.io import fits
    import numpy as np
    
    xml_dir = 'XML_models'
    srcmap_dir = 'Source_Maps'
    fit_xml_dir = 'Fitted_XML_models'
    
    if not os.path.exists(fit_xml_dir):
        os.makedirs(fit_xml_dir, exist_ok=True)
        
    xml_file = os.path.join(xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}.xml')
    srcmap_file = os.path.join(srcmap_dir, f'GCE_17yr_srcmap_Model_{roman_name}.fits')
    
    # 병합된 결과를 저장할 새로운 임시 최적화 파일 경로
    merged_xml_file = os.path.join(fit_xml_dir, f'GCE_17yr_FL16Y_Model_{roman_name}_merged.xml')
    merged_srcmap_file = os.path.join(fit_xml_dir, f'GCE_17yr_srcmap_Model_{roman_name}_merged.fits')
    out_xml_file = os.path.join(fit_xml_dir, f'GCE_17yr_fitted_Model_{roman_name}.xml')
    
    if not os.path.exists(srcmap_file):
        return (roman_name, None, "Source map 파일 누락")
        
    try:
        # =========================================================
        # [핵심 최적화] 고정된 소스들을 단일 맵으로 병합 (Memory Diet)
        # =========================================================
        # 이미 병합된 파일이 없다면 새롭게 병합을 수행합니다.
        if not os.path.exists(merged_xml_file) or not os.path.exists(merged_srcmap_file):
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # 1. XML에서 완전히 고정된(free="0") 소스 이름 추출
            fixed_source_names = set()
            for source in root.findall('source'):
                name = source.get('name')
                # 은하 배경, 등방성 배경 등은 제외하고 순수 점/확장 광원만 대상
                if any(x in name for x in ['Pi0', 'Bremss', 'ICS', 'iso_p8v3']):
                    continue
                    
                spectrum = source.find('spectrum')
                if spectrum is not None:
                    is_fixed = True
                    for param in spectrum.findall('parameter'):
                        if str(param.get('free')) == '1':
                            is_fixed = False
                            break
                    if is_fixed:
                        fixed_source_names.add(name)
                        
            # 2. Source Map FITS 파일을 열어서 고정 소스들의 맵 데이터 합치기
            with fits.open(srcmap_file) as hdul:
                new_hdul = fits.HDUList([hdul[0].copy()])
                sum_data = None
                sample_header = None
                merged_count = 0
                actually_merged = set()
                
                for hdu in hdul[1:]:
                    # FITS 파일에 이미 생성되어 있는 고정 소스라면 Array를 합칩니다.
                    if isinstance(hdu, fits.ImageHDU) and hdu.name in fixed_source_names:
                        if sum_data is None:
                            sum_data = np.zeros_like(hdu.data, dtype=np.float32)
                            sample_header = hdu.header.copy()
                        if hdu.data is not None:
                            sum_data += hdu.data
                        actually_merged.add(hdu.name)
                        merged_count += 1
                    else:
                        # 자유 파라미터가 있거나, 시스템 확장자(ENERGIES 등)는 그대로 유지
                        new_hdul.append(hdu.copy())
                
                # 병합된 단일 HDU 생성 및 FITS에 추가
                if sum_data is not None:
                    sample_header['EXTNAME'] = 'MergedFixed'
                    merged_hdu = fits.ImageHDU(data=sum_data, header=sample_header, name='MergedFixed')
                    new_hdul.append(merged_hdu)
                    
                new_hdul.writeto(merged_srcmap_file, overwrite=True)
                
            # 3. XML 파일 업데이트 (합쳐진 소스들은 개별 리스트에서 삭제하고 단일 Diffuse 소스로 추가)
            for source in root.findall('source'):
                if source.get('name') in actually_merged:
                    root.remove(source)
                    
            if merged_count > 0:
                merged_src = ET.SubElement(root, 'source', {'name': 'MergedFixed', 'type': 'DiffuseSource'})
                spec = ET.SubElement(merged_src, 'spectrum', {'type': 'ConstantValue'})
                ET.SubElement(spec, 'parameter', {'name': 'Value', 'value': '1.0', 'free': '0', 'min': '0.1', 'max': '10.0', 'scale': '1.0'})
                spatial = ET.SubElement(merged_src, 'spatialModel', {'type': 'MapCubeFunction', 'file': 'dummy_merged.fits'})
                ET.SubElement(spatial, 'parameter', {'name': 'Normalization', 'value': '1.0', 'free': '0', 'min': '0.1', 'max': '10.0', 'scale': '1.0'})
                
            tree.write(merged_xml_file, encoding='utf-8', xml_declaration=True)
            print(f"  [최적화 완료] Model {roman_name}: {merged_count}개의 고정 소스를 단일 배경으로 압축했습니다!")

        # =========================================================
        # 4. 피팅 실행 (병합되어 엄청나게 가벼워진 최적화 파일 사용)
        # =========================================================
        obs = ba.BinnedObs(srcMaps=merged_srcmap_file,
                           expCube='GCE_17yr_ltcube.fits',
                           binnedExpMap='GCE_17yr_expcube_large.fits',
                           irfs='P8R3_ULTRACLEANVETO_V3')
                           
        like = ba.BinnedAnalysis(obs, srcModel=merged_xml_file, optimizer='NewMinuit')
        likeObj = like.fit(covar=True, tol=1e-2, verbosity=0)
        
        logL = like.logLike.value()
        like.logLike.writeXml(out_xml_file)
        
        return (roman_name, logL, "Success")
        
    except Exception as e:
        return (roman_name, None, f"Error: {e}")

# =====================================================================
# 2. 메인 실행 블록 (멀티프로세싱 코어 할당 및 관리)
# =====================================================================
if __name__ == '__main__':
    start_time = time.time()
    
    xml_dir = 'XML_models'
    srcmap_dir = 'Source_Maps'
    fit_xml_dir = 'Fitted_XML_models'

    if not os.path.exists(fit_xml_dir):
        os.makedirs(fit_xml_dir)

    naming_conv_file = 'NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
    models_info = []
    with open(naming_conv_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 2:
                models_info.append(parts[0])

    # 💡 램 사용량이 획기적으로 줄었으므로 32코어로 마음껏 돌리셔도 안전합니다!
    NUM_CORES = 32  
    print(f"▶️ 총 {len(models_info)}개의 모델에 대해 {NUM_CORES}개의 코어를 이용한 [최적화] 병렬 피팅을 시작합니다...")

    results_file = 'Likelihood_Results_17yr.csv'
    with open(results_file, 'w') as f:
        f.write("Model,LogLikelihood,Status\n")

    with mp.Pool(processes=NUM_CORES) as pool:
        for result in pool.imap_unordered(run_fitting_for_model, models_info):
            roman_name, logL, status = result
            
            if status == "Success":
                print(f" ✅ [완료] Model {roman_name} | -log(L) = {logL:.2f}")
            else:
                print(f" ❌ [실패] Model {roman_name} | 이유: {status}")
                
            with open(results_file, 'a') as f:
                f.write(f"{roman_name},{logL},{status}\n")

    end_time = time.time()
    print(f"\n🎉 병렬 피팅 대장정이 모두 완료되었습니다!")
    print(f"⏱️ 총 소요 시간: {(end_time - start_time)/3600:.2f} 시간")

In [2]:
"""
make_mask_plot.py -- FL16Y-based energy-dependent mask generation and visualization

Usage:
    python3 make_mask_plot.py

Output:
    mask_plot.png     -- mask visualization for 3 energy bins
    masks_fl16y.fits  -- full energy-bin masks (can be used in fitting)

Requirements:
    pip install numpy matplotlib astropy scipy --break-system-packages
    XML_models/GCE_17yr_FL16Y_Model_I.xml  (FL16Y source positions)
"""

import os
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xml.etree.ElementTree as ET
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u

# =====================================================================
# Settings
# =====================================================================
BASE_DIR     = os.path.abspath('.')
XML_FILE     = os.path.join(BASE_DIR, 'XML_models', 'GCE_17yr_FL16Y_Model_I.xml')
OUTPUT_PNG   = os.path.join(BASE_DIR, 'mask_plot.png')
OUTPUT_FITS  = os.path.join(BASE_DIR, 'masks_fl16y.fits')

# Map settings (same as ccube)
NX           = 400       # pixels (40 deg ROI)
NY           = 400
PIXEL_DEG    = 0.1       # deg per pixel

# Energy bins (300 MeV ~ 1 TeV, 17 log bins)
E_MIN        = 300.0     # MeV
E_MAX        = 1_000_000.0
N_BINS       = 17
ENERGIES     = np.logspace(np.log10(E_MIN), np.log10(E_MAX), N_BINS + 1)
E_CENTERS    = np.sqrt(ENERGIES[:-1] * ENERGIES[1:])  # geometric mean

# Masking parameters
LAT_CUT          = 2.0   # Galactic plane mask |b| < 2 deg
TS_BRIGHT        = 49    # Bright source TS threshold
TS_MIN           = 25    # Minimum TS for masking
THETA_S_SCALE    = 0.5   # standard source radius = 0.5 x PSF_68
THETA_L_SCALE    = 1.0   # bright source radius = 1.0 x PSF_68
ROI_RADIUS       = 20.0  # ROI radius (deg)


# =====================================================================
# PSF 68% containment radius (empirical)
# =====================================================================
def psf_68(energy_mev):
    """P8R3 FRONT+BACK PSF 68% radius (deg)"""
    c0, beta, c1 = 3.5, 0.8, 0.1
    return np.sqrt((c0 * (energy_mev / 100.0)**(-beta))**2 + c1**2)


# =====================================================================
# Load source positions from FL16Y XML
# =====================================================================
def load_fl16y_sources(xml_path, roi_radius=20.0):
    """
    Read FL16Y point source positions and approximate TS from XML.
    FL16Y XML has no TS, so brightness is approximated by source type.
    all FL16Y sources treated as TS >= 25.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    sources = []
    for src in root.findall('source'):
        name = src.get('name', '')
        if 'FL16Y' not in name and 'FGES' not in name:
            continue

        # Extract RA/Dec
        spatial = src.find('spatialModel')
        if spatial is None:
            continue

        ra_el  = spatial.find(".//parameter[@name='RA']")
        dec_el = spatial.find(".//parameter[@name='DEC']")
        if ra_el is None or dec_el is None:
            continue

        ra  = float(ra_el.get('value', 0))
        dec = float(dec_el.get('value', 0))

        # Convert to galactic coordinates
        coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
        l = coord.galactic.l.degree
        b = coord.galactic.b.degree
        if l > 180:
            l -= 360.0

        # Remove sources outside ROI
        if abs(l) > roi_radius or abs(b) > roi_radius:
            continue

        # FL16Y sources all have TS > 25 (catalog threshold)
        # FGES are extended sources, treated as brighter
        is_bright = 'FGES' in name
        ts_approx = 100.0 if is_bright else 30.0

        sources.append({
            'name'     : name,
            'l'        : l,
            'b'        : b,
            'ts'       : ts_approx,
            'is_bright': is_bright,
        })

    return sources


# =====================================================================
# Generate energy-dependent masks
# =====================================================================
def make_masks(sources, nx, ny, pixel_deg, energy_centers,
               lat_cut, ts_bright, ts_min,
               theta_s_scale, theta_l_scale):

    n_bins = len(energy_centers)
    masks  = np.ones((n_bins, ny, nx), dtype=np.float32)

    crpix_x = nx / 2.0 + 0.5
    crpix_y = ny / 2.0 + 0.5

    # ── (a) Galactic plane masking ────────────────────────────────
    x_idx = np.arange(nx)
    y_idx = np.arange(ny)
    XX, YY = np.meshgrid(x_idx, y_idx)
    lat_map = (YY - crpix_y + 1) * pixel_deg     # b (deg)
    gal_mask = np.abs(lat_map) < lat_cut
    for ie in range(n_bins):
        masks[ie][gal_mask] = 0.0
    print(f"  Galactic plane masked: |b| < {lat_cut} deg")

    # ── (b) Point source masking ────────────────────────────────
    # Convert to pixel coordinates
    Y_grid, X_grid = np.ogrid[:ny, :nx]

    mask_sources = [s for s in sources if s['ts'] >= ts_min]
    n_bright = sum(1 for s in mask_sources if s['is_bright'])
    print(f"  Point sources masked: {len(mask_sources)} "
          f"(FGES/bright {n_bright}, FL16Y {len(mask_sources)-n_bright})")

    for ie, e_center in enumerate(energy_centers):
        psf = psf_68(e_center)
        for src in mask_sources:
            # Pixel coordinates
            px = (src['l'] / (-pixel_deg)) + crpix_x - 1
            py = (src['b'] /  pixel_deg)  + crpix_y - 1

            if not (0 <= px < nx and 0 <= py < ny):
                continue

            scale = theta_l_scale if src['is_bright'] else theta_s_scale
            r_pix = (scale * psf) / pixel_deg
            dist2 = (X_grid - px)**2 + (Y_grid - py)**2
            masks[ie][dist2 <= r_pix**2] = 0.0

    return masks


# =====================================================================
# Visualization
# =====================================================================
def plot_masks(masks, energy_centers, sources, pixel_deg,
               output_path, lat_cut):

    # Select energy bins to visualize: low / mid / high
    n_bins  = len(energy_centers)
    ie_list = [0, n_bins // 2, n_bins - 1]
    titles  = [
        f"Low Energy\n{energy_centers[ie_list[0]]/1e3:.2f} GeV",
        f"Mid Energy\n{energy_centers[ie_list[1]]/1e3:.2f} GeV",
        f"High Energy\n{energy_centers[ie_list[2]]/1e3:.2f} GeV",
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                             facecolor='#0d1117')
    fig.suptitle(
        f"Fermi-LAT GCE 17.5yr — Energy-Dependent Mask (FL16Y)\n"
        f"|b| < {lat_cut}° Galactic Plane + FL16Y Point Source PSF Masking",
        color='white', fontsize=13, y=1.01
    )

    extent = [ROI_RADIUS, -ROI_RADIUS, -ROI_RADIUS, ROI_RADIUS]

    for col, (ie, title) in enumerate(zip(ie_list, titles)):
        ax = axes[col]
        ax.set_facecolor('#0d1117')

        # Mask image (white=used, dark=masked)
        cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
            'mask', ['#1a1a2e', '#e8e8e8'])
        ax.imshow(masks[ie], origin='lower', cmap=cmap,
                  extent=extent, vmin=0, vmax=1, aspect='equal')

        # Galactic plane boundary
        ax.axhline(y= lat_cut, color='#ff6b35', lw=0.8, ls='--', alpha=0.7)
        ax.axhline(y=-lat_cut, color='#ff6b35', lw=0.8, ls='--', alpha=0.7)

        # ROI circle
        theta_arr = np.linspace(0, 2*np.pi, 300)
        ax.plot(ROI_RADIUS * np.cos(theta_arr),
                ROI_RADIUS * np.sin(theta_arr),
                color='#4fc3f7', lw=0.8, ls=':', alpha=0.5)

        # Mark bright sources (FGES)
        psf = psf_68(energy_centers[ie])
        for src in sources:
            if not src['is_bright']:
                continue
            if abs(src['l']) > ROI_RADIUS or abs(src['b']) > ROI_RADIUS:
                continue
            r = THETA_L_SCALE * psf
            circ = plt.Circle((-src['l'], src['b']), r,
                               color='#ff6b35', fill=False,
                               lw=0.8, alpha=0.8)
            ax.add_patch(circ)

        # Statistics
        frac_mask = np.sum(masks[ie] == 0) / masks[ie].size * 100
        frac_use  = 100 - frac_mask

        ax.set_title(title, color='white', fontsize=11, pad=8)
        ax.set_xlabel('l (deg)', color='#aaaaaa', fontsize=9)
        if col == 0:
            ax.set_ylabel('b (deg)', color='#aaaaaa', fontsize=9)

        ax.tick_params(colors='#aaaaaa', labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')

        # Masking fraction text
        ax.text(0.03, 0.97,
                f"PSF68 = {psf:.2f} deg\n"
                f"Used: {frac_use:.1f}%\n"
                f"Masked: {frac_mask:.1f}%",
                transform=ax.transAxes,
                color='#4fc3f7', fontsize=8, va='top',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='#0d1117', alpha=0.8,
                          edgecolor='#333333'))

        ax.set_xlim( ROI_RADIUS + 1, -ROI_RADIUS - 1)
        ax.set_ylim(-ROI_RADIUS - 1,  ROI_RADIUS + 1)

    # Fix colorbar — separate axis on the right
    plt.tight_layout(rect=[0, 0, 0.92, 1])
    cbar_ax = fig.add_axes([0.93, 0.12, 0.015, 0.76])
    sm = plt.cm.ScalarMappable(
        cmap=cmap,
        norm=mcolors.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Mask value (1=used, 0=masked)',
                   color='#aaaaaa', fontsize=9)
    cbar.ax.yaxis.set_tick_params(color='#aaaaaa', labelcolor='#aaaaaa')
    plt.savefig(output_path, dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    print(f"\n  Saved: {output_path}")
    plt.close()


# =====================================================================
# Main
# =====================================================================
if __name__ == '__main__':
    print("=" * 60)
    print("FL16Y-based Energy-Dependent Mask Generation")
    print("=" * 60)

    # Load FL16Y sources
    if not os.path.exists(XML_FILE):
        print(f"ERROR: XML file not found: {XML_FILE}")
        sys.exit(1)

    print(f"\n[1] Loading FL16Y sources: {XML_FILE}")
    sources = load_fl16y_sources(XML_FILE, roi_radius=ROI_RADIUS)
    n_fl16y = sum(1 for s in sources if 'FL16Y' in s['name'])
    n_fges  = sum(1 for s in sources if 'FGES'  in s['name'])
    print(f"  Sources in ROI: {len(sources)} "
          f"(FL16Y {n_fl16y}, FGES {n_fges})")

    # Generate masks
    print(f"\n[2] Generating energy-dependent masks ({N_BINS} bins)...")
    masks = make_masks(
        sources, NX, NY, PIXEL_DEG, E_CENTERS,
        LAT_CUT, TS_BRIGHT, TS_MIN,
        THETA_S_SCALE, THETA_L_SCALE
    )

    # Mask save
    print(f"\n[3] Saving FITS: {OUTPUT_FITS}")
    hdr = fits.Header()
    hdr['NAXIS1']  = NX
    hdr['NAXIS2']  = NY
    hdr['NAXIS3']  = N_BINS
    hdr['PIXDEG']  = PIXEL_DEG
    hdr['LATCUT']  = LAT_CUT
    hdr['TSBRIGHT']= TS_BRIGHT
    hdr['TSMIN']   = TS_MIN
    hdr['THETAS']  = THETA_S_SCALE
    hdr['THETAL']  = THETA_L_SCALE
    fits.PrimaryHDU(masks, header=hdr).writeto(OUTPUT_FITS, overwrite=True)

    # Visualization
    print(f"\n[4] Generating mask visualization...")
    for ie in [0, N_BINS // 2, N_BINS - 1]:
        frac = np.sum(masks[ie] == 0) / (NX * NY) * 100
        psf  = psf_68(E_CENTERS[ie])
        print(f"  Bin {ie:2d} ({E_CENTERS[ie]/1e3:.2f} GeV): "
              f"Masked {frac:.1f}%, PSF68={psf:.2f} deg")

    plot_masks(masks, E_CENTERS, sources, PIXEL_DEG,
               OUTPUT_PNG, LAT_CUT)

    print(f"\n{'='*60}")
    print(f"Done!")
    print(f"  Mask image: {OUTPUT_PNG}")
    print(f"  Mask FITS : {OUTPUT_FITS}")
    print(f"{'='*60}")

FL16Y-based Energy-Dependent Mask Generation

[1] Loading FL16Y sources: /home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data/XML_models/GCE_17yr_FL16Y_Model_I.xml
  Sources in ROI: 670 (FL16Y 669, FGES 1)

[2] Generating energy-dependent masks (17 bins)...
  Galactic plane masked: |b| < 2.0 deg
  Point sources masked: 670 (FGES/bright 1, FL16Y 669)

[3] Saving FITS: /home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data/masks_fl16y.fits

[4] Generating mask visualization...
  Bin  0 (0.38 GeV): Masked 38.0%, PSF68=1.20 deg
  Bin  8 (17.32 GeV): Masked 10.3%, PSF68=0.11 deg
  Bin 16 (787.75 GeV): Masked 10.2%, PSF68=0.10 deg

  Saved: /home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data/mask_plot.png

Done!
  Mask image: /home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data/mask_plot.png
  Mask FITS : /home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data/masks_fl16y.fits
